# Netflix-Style Recommender — Training (Colab)

Trains SVD + Content + NCF + Hybrid on MovieLens 1M, logs to W&B, and pushes models to Hugging Face Hub.

**Before running:** Runtime → Change runtime type → **GPU**.

You'll need:
- A Hugging Face token (write scope) → https://huggingface.co/settings/tokens
- A W&B API key → https://wandb.ai/authorize

## 1. Clone the repo

In [ ]:
!git clone https://github.com/nehashirodkar/netflix-content-recommendation-engine.git
%cd netflix-content-recommendation-engine

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Authenticate (HF + W&B)

In [ ]:
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face token: ')
os.environ['HF_MODEL_REPO'] = input('HF model repo (e.g. NehaS98/netflix-recsys-models): ').strip() or 'NehaS98/netflix-recsys-models'
os.environ['WANDB_API_KEY'] = getpass.getpass('W&B API key (leave empty to disable): ')
if not os.environ['WANDB_API_KEY']:
    os.environ['WANDB_MODE'] = 'disabled'

## 4. Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 5. Train all models + evaluate + push to HF Hub

In [ ]:
!python -m src.training.train_all --push-hf

## 6. (Optional) Inspect saved metrics

In [ ]:
import json
print(json.dumps(json.load(open('models/metadata.json')), indent=2))

## 7. (Optional) Sanity-check inference

In [ ]:
from src.api.recommender import Recommender
rec = Recommender().load()
print(rec.recommend(model='hybrid', user_id=1, k=5))
print(rec.recommend(model='content', history_movie_ids=[1, 2, 3], k=5))